# RAG Evaluation Metrics

Metrics that assess the quality of a RAG pipeline across three independent dimensions.


## The RAG Triad

Every RAG failure falls into one of three buckets. Measure all three independently — a single "quality" score hides which component is broken.

```
Query ──► Retriever ──► Context chunks ──► Generator ──► Answer
              │                                │
       Context Relevance               Faithfulness
                                    Answer Relevancy
```

---

### 1. Faithfulness (Groundedness)
**Evaluates:** Does every claim in the answer come from the retrieved context — or did the model hallucinate?

- Decompose the answer into individual atomic claims
- For each claim, ask: *"Is this claim supported by the context?"*
- Score = `supported claims / total claims`
- **Target:** ≥ 0.80
- **Low score means:** Generator is hallucinating. Fix: tighten system prompt, lower temperature, add citation constraints.

---

### 2. Answer Relevancy
**Evaluates:** Does the answer actually address what the user asked?

- Measures semantic alignment between the question and the answer
- A faithful answer can still be irrelevant if it answers the wrong question
- Score = cosine similarity between question embedding and back-generated questions from the answer (or keyword overlap as a proxy)
- **Target:** ≥ 0.80
- **Low score means:** Answer is factual but off-topic. Fix: refine system prompt, improve query reformulation.

---

### 3. Context Relevance (Context Precision)
**Evaluates:** Were the retrieved chunks actually useful for answering the query?

- For each retrieved chunk: *"Does this chunk contain information relevant to the query?"*
- Score = `relevant chunks / total chunks retrieved`
- **Target:** ≥ 0.70
- **Low score means:** Retriever is fetching noisy/wrong chunks. Fix: embedding model, chunk size, top-k, re-ranking.

---

## Diagnosis Map

| Low Metric | Root Cause | Where to Fix |
|------------|-----------|--------------|
| Context Relevance | Wrong chunks fetched | Retriever — embedding model, chunk size, top-k |
| Faithfulness | Model hallucinating despite good context | Generator — prompt, temperature, citations |
| Answer Relevancy | Answer is factual but off-topic | Alignment — system prompt, query rewriting |

---

## Additional RAG Metrics (Ragas / DeepEval)

| Metric | Evaluates |
|--------|-----------|
| **Contextual Recall** | How much of the ground-truth answer is covered by the retrieved context? Catches retriever gaps. |
| **Contextual Precision** | Are the most relevant chunks ranked highest? Penalises noisy chunks retrieved before good ones. |
| **Noise Robustness** | Does the answer degrade when irrelevant chunks are mixed in with relevant ones? |
| **Negative Rejection** | Does the model correctly say *"I don't know"* when the context contains no answer? |
| **Counterfactual Robustness** | Does the model resist context chunks that contain deliberately wrong information? |
| **Information Integration** | Can the model synthesise a correct answer from facts spread across multiple chunks? |


## Retrieval Ranking Metrics

Relevance alone isn't enough — **the right chunk must appear early**. A system that retrieves the correct chunk at rank 10 is far worse than one that retrieves it at rank 1. These metrics measure ranking quality.

| Metric | Evaluates | Formula | Target |
|--------|-----------|---------|--------|
| **Precision@K** | Of the top-K retrieved chunks, what fraction are actually relevant? | `relevant in top-K / K` | ≥ 0.70 |
| **Recall@K** | Of all relevant chunks that exist, how many were retrieved in the top-K? | `relevant retrieved / total relevant` | ≥ 0.80 |
| **MRR** (Mean Reciprocal Rank) | How quickly does the first relevant chunk appear? | `avg(1 / rank of first relevant doc)` | ≥ 0.80 → relevant doc is at rank 1–2 |
| **NDCG** (Normalised Discounted Cumulative Gain) | Relevance weighted by rank position — penalises burying good chunks lower in the list | logarithmic position discount | ≥ 0.80 |

**When to use which:**
- Use **Precision@K** when you pass a fixed number of chunks to the generator (top-K is your context budget)
- Use **Recall@K** when missing a relevant chunk is more costly than including noise
- Use **MRR** when there's typically one best answer chunk and rank-1 matters most
- Use **NDCG** when partial relevance exists (chunks vary in quality, not just relevant/not)

---

## Context Neglect vs Hallucination

Two distinct failure modes that look similar on the surface but have different fixes:

```
Hallucination    → Context is present, model INVENTS facts not in it
                   Detected by: faithfulness score < 0.8
                   Fix: lower temperature, stricter grounding prompt, citation constraints

Context Neglect  → Context is present and correct, model IGNORES it
                   and answers from training memory instead
                   Detected by: remove context entirely — if answer doesn't change,
                                the model was ignoring retrieval
                   Fix: stronger grounding instruction in system prompt,
                        RAG-specific prompt template ("Answer ONLY using the context below")
```

> Key test: Run the same query with and without the retrieved context. If the answer is identical, the model is not using retrieval at all — you have a context neglect problem, not a retrieval problem.

---

## Context Window Realism

**Evaluating with more context than production uses overestimates real performance by 25–30%.**

Always evaluate under the same token constraints your production system enforces.

```
Wrong:  Evaluate with all 50 retrieved chunks (never happens in prod)
Right:  Evaluate with the same truncated context your production prompt uses

Rule:   MAX_CONTEXT_TOKENS in eval == MAX_CONTEXT_TOKENS in production
```

This matters because models perform significantly better with more context available — your eval scores will be inflated if you're not enforcing the same limits.
